In [1]:
# Solo si NO se puede clonar el repositorio: subir CREDIT_repo.zip al panel de archivos y quitar el '#'.
# En nuestra corrida lo usamos porque la carpeta final/ todavía no estaba en GitHub.
# !unzip -q CREDIT_repo.zip

# SI4006 · Proyecto final — Técnica de eficiencia: cuantizar el generador

**Qué hace este notebook.** Toma el sistema final de M3 (RAG híbrido + reranking + calculadora con router) y le cambia
solo una pieza: el generador `Qwen2.5-1.5B-Instruct`, que es el componente que más tarda (la recuperación completa
toma unos 47 ms; la generación, más de un segundo en GPU y bastante más en CPU). Lo comparamos en tres versiones:

| Versión | Pesos | Motor |
| :-- | :-- | :-- |
| **Antes** · fp32 | precisión completa | transformers |
| **Después** · Q8_0 | int8 | llama.cpp (GGUF) |
| **Después** · Q4_K_M | int4 (k-quants) | llama.cpp (GGUF) |

**Misma máquina para todo.** Las tres versiones generan en la **CPU** del entorno de Colab, porque es el hardware al que
vamos a desplegar (Hugging Face Spaces gratuito: 2 vCPU, 16 GB de RAM, sin GPU). La GPU solo se usa en la fase B, para
el juez del harness, cuando ya no se mide latencia.

**Misma vara.** Mismo eval set de M3 (`eval_set_m3_v2.json`, 15 casos), mismo harness de tres dimensiones, mismo juez
(Qwen2.5-7B en 4 bits) y mismo prompt. El texto de entrada al generador es idéntico en las tres versiones: llama.cpp recibe
los tokens producidos por la plantilla de chat del tokenizer original.

**Cómo correrlo.** Colab → *Entorno de ejecución* → *Cambiar tipo* → **T4 GPU**. *Ejecutar todo*. Tarda alrededor de una
hora; la primera vez descarga ~20 GB de pesos.

## 0 · El criterio, escrito antes de medir

El enunciado pide decidir **antes** de medir cuánta calidad estamos dispuestos a perder. Este es nuestro criterio. Se
fija en la celda siguiente y se verifica de forma automática al final del notebook; la celda no se toca después de ver
los números (el historial de git lo muestra).

In [2]:
# Criterio pre-registrado. Se escribe ANTES de correr el resto del notebook.
CRITERIO = {
    'max_perdida_juez_v11': 0.3,    # puntos de D2 (juez v1.1, escala 1-5) que aceptamos perder
    'max_perdida_aciertos': 1,      # aciertos de D3 (sobre 15) que aceptamos perder
    'valvula_intacta': True,        # 2/2 en el eval set: no aceptamos perder ninguno
    'max_perdida_adversariales': 0, # pruebas adversariales que aceptamos perder
    'min_reduccion_p50': 0.40,      # a cambio, la latencia p50 en CPU debe bajar al menos 40 %
    'min_reduccion_memoria': 0.50,  # y el tamaño del generador al menos 50 %
}
CRITERIO_TEXTO = (
    'Aceptamos perder hasta {max_perdida_juez_v11} puntos del juez v1.1 y hasta {max_perdida_aciertos} acierto de '
    'dominio, sin perder ningún caso de válvula ni ninguna prueba adversarial, si la latencia p50 en CPU baja al menos '
    '{p50:.0%} y el tamaño del generador al menos {mem:.0%} frente a la versión fp32.'
).format(**CRITERIO, p50=CRITERIO['min_reduccion_p50'], mem=CRITERIO['min_reduccion_memoria'])
print(CRITERIO_TEXTO)

Aceptamos perder hasta 0.3 puntos del juez v1.1 y hasta 1 acierto de dominio, sin perder ningún caso de válvula ni ninguna prueba adversarial, si la latencia p50 en CPU baja al menos 40% y el tamaño del generador al menos 50% frente a la versión fp32.


## 1 · Setup

In [3]:
%pip install -q sentence-transformers rank_bm25 bitsandbytes accelerate psutil huggingface_hub

# llama.cpp: primero la rueda precompilada para CPU; si no hay para esta versión de Python, pip lo compila
# (tarda 10-15 minutos adicionales, pero funciona en Colab).
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python', '--prefer-binary',
                    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cpu'],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:])
import llama_cpp; print('llama-cpp-python', llama_cpp.__version__)

 
llama-cpp-python 0.3.36


In [4]:
import os, sys, json, time, gc, random, platform
from pathlib import Path

REPO_URL = 'https://github.com/CamsMars/CREDIT'
if not Path('CREDIT').exists() and not Path('final/space/credit_rag.py').exists():
    !git clone -q {REPO_URL}
RAIZ = Path('CREDIT') if Path('CREDIT').exists() else Path('.')
sys.path[:0] = [str(RAIZ / 'final' / 'space'), str(RAIZ / 'final')]

import numpy as np, pandas as pd, torch, psutil, transformers
import credit_rag as cr
import harness_final as hf

random.seed(cr.SEMILLA); np.random.seed(cr.SEMILLA); torch.manual_seed(cr.SEMILLA)
N_CPU = os.cpu_count()
torch.set_num_threads(N_CPU)
SALIDA = RAIZ / 'final' / 'salida'; SALIDA.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 80)

MAQUINA = {'cpu': platform.processor() or platform.machine(), 'n_cpu': N_CPU,
           'ram_gb': round(psutil.virtual_memory().total / 1e9, 1),
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(MAQUINA)
assert torch.cuda.is_available(), 'Activa la GPU T4: la necesita el juez en la fase B.'

{'cpu': 'x86_64', 'n_cpu': 2, 'ram_gb': 13.6, 'gpu': 'Tesla T4'}


In [5]:
# Los archivos GGUF existen con esos nombres en el repositorio oficial de Qwen.
from huggingface_hub import list_repo_files
disponibles = set(list_repo_files(cr.GGUF_REPO))
for v, f in cr.GGUF_ARCHIVOS.items():
    print(f'{v:7s} {f:45s}', 'OK' if f in disponibles else 'NO EXISTE')
assert all(f in disponibles for f in cr.GGUF_ARCHIVOS.values()), sorted(disponibles)

q8_0    qwen2.5-1.5b-instruct-q8_0.gguf               OK
q4_k_m  qwen2.5-1.5b-instruct-q4_k_m.gguf             OK


## 2 · Eval set, pruebas de robustez e índice (compartido por las tres versiones)

In [6]:
with open(RAIZ / 'M3' / 'm3_salida' / 'eval_set_m3_v2.json', encoding='utf-8') as f:
    eval_set = json.load(f)
print(len(eval_set), 'casos del eval set de M3 |', len(hf.PRUEBAS_ADVERSARIALES), 'adversariales |',
      len(hf.PARES_INVARIANZA), 'pares de invarianza')

# El índice va en CPU, igual que en el Space.
indice = cr.Indice(RAIZ / 'final' / 'space' / 'corpus_m3.json', device='cpu')
print(len(indice.corpus), 'documentos ->', len(indice.chunks), 'chunks')

15 casos del eval set de M3 | 5 adversariales | 5 pares de invarianza


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

8 documentos -> 23 chunks


In [7]:
# Verificación: la búsqueda densa exacta en numpy devuelve el mismo orden que Chroma en M3.
VERIFICAR_CHROMA = True
if VERIFICAR_CHROMA:
    try:
        %pip install -q chromadb
        import chromadb
        col = chromadb.Client().create_collection('verif', metadata={'hnsw:space': 'cosine'})
        col.add(ids=indice.ids, documents=indice.chunks, embeddings=indice.st.encode(indice.chunks).tolist())
        iguales = 0
        for c in eval_set:
            r = col.query(query_embeddings=indice.st.encode([c['input']]).tolist(), n_results=10)['ids'][0]
            iguales += r == [p['id'] for p in indice.buscar_densa(c['input'], 10)]
        print(f'Top-10 idéntico a Chroma en {iguales}/{len(eval_set)} consultas')
    except Exception as e:
        print('No se pudo verificar con Chroma:', type(e).__name__, str(e)[:200])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Top-10 idéntico a Chroma en 15/15 consultas


## 3 · Fase A — correr las tres versiones en CPU

Para cada versión: se carga el generador, se mide la memoria que ocupa, se hace una pregunta de calentamiento que no está
en el eval set (la primera llamada siempre es más lenta y no la queremos en las percentiles), y se corren el eval set,
las pruebas adversariales y las paráfrasis de invarianza. Las respuestas se guardan en `final/salida/crudos_<versión>.json`.

In [8]:
VERSIONES = ['fp32', 'q8_0', 'q4_k_m']
proc = psutil.Process()
crudos, recursos = {}, {}

for v in VERSIONES:
    gc.collect()
    rss0 = proc.memory_info().rss
    gen = cr.crear_generador(v, device='cpu') if v.startswith('fp') else cr.crear_generador(v, n_threads=N_CPU)
    sistema = cr.SistemaCredit(indice, gen)
    t0 = time.perf_counter(); sistema.responder('¿Qué modelo base usa el sistema?'); calentamiento = time.perf_counter() - t0
    rss1 = proc.memory_info().rss
    print(f'\n[{v}] {gen.nombre} | carga {gen.segundos_carga:.1f} s | calentamiento {calentamiento:.1f} s')

    t0 = time.perf_counter()
    ev = hf.correr(sistema, eval_set)
    adv = hf.correr(sistema, hf.PRUEBAS_ADVERSARIALES)
    inv = hf.correr(sistema, hf.PARES_INVARIANZA)
    crudos[v] = {'eval': ev, 'adversariales': adv, 'invarianza': inv}
    lat = hf.latencias(ev)
    recursos[v] = {'generador': gen.nombre, 'tamano_gb': round((gen.tamano_bytes or float('nan')) / 1e9, 2),
                   'ram_extra_gb': round((rss1 - rss0) / 1e9, 2), 'carga_s': round(gen.segundos_carga, 1), **lat}
    print(f'   eval p50 {lat["p50"]:.2f} s · p95 {lat["p95"]:.2f} s · total {time.perf_counter()-t0:.0f} s | '
          f'tamaño {recursos[v]["tamano_gb"]} GB · RAM extra {recursos[v]["ram_extra_gb"]} GB')
    with open(SALIDA / f'crudos_{v}.json', 'w', encoding='utf-8') as f:
        json.dump(crudos[v], f, ensure_ascii=False, indent=1)
    del sistema, gen; gc.collect()

with open(SALIDA / 'recursos.json', 'w') as f:
    json.dump({'maquina': MAQUINA, 'recursos': recursos}, f, indent=1)
pd.DataFrame(recursos).T

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (7 outputs left out)

qwen2.5-1.5b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 1.12GB            

qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            


[q4_k_m] Qwen/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf (llama.cpp, cpu) | carga 13.3 s | calentamiento 41.6 s
   eval p50 43.07 s · p95 131.11 s · total 1386 s | tamaño 1.12 GB · RAM extra 1.79 GB


,generador,tamano_gb,ram_extra_gb,carga_s,p50,p95,media,max
fp32,"Qwen/Qwen2.5-1.5B-Instruct (float32, cpu)",6.17,6.32,46.6,44.187,123.5131,63.953533,127.193
q8_0,"Qwen/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q8_0.gguf (llama.cpp, ...",1.89,1.63,31.9,62.567,178.8978,81.075867,181.261
q4_k_m,Qwen/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf (llama.cpp...,1.12,1.79,13.3,43.073,131.1109,59.388,133.766


## 4 · Fase B — calificar con el harness (GPU)

Mismo juez de M3 (Qwen2.5-7B-Instruct en 4 bits) y la vara original de M2 como columna aparte. El juez de M2 se fija en
Qwen2.5-1.5B fp16 para las tres filas: si usáramos el generador de cada versión, la vara cambiaría junto con lo medido.

In [9]:
juez = hf.JuezHF()
juez_m2 = cr.GeneradorHF(dtype=torch.float16, device='cuda')
print('Juez v1.1:', juez.nombre, '| juez M2:', juez_m2.nombre)

cordura = hf.prueba_cordura(juez, juez_m2)
print(cordura)

hf.CONTADOR.update(n=0, total=0, deterministas=0)
NOMBRES = {'fp32': 'Antes · fp32 (sistema M3)', 'q8_0': 'Después · Q8_0 (int8)', 'q4_k_m': 'Después · Q4_K_M (int4)'}
scores = {v: hf.calificar(crudos[v]['eval'], eval_set, juez, juez_m2, indice.st, NOMBRES[v]) for v in VERSIONES}
print('Llamadas al juez sin veredicto legible:', hf.CONTADOR)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Juez v1.1: Qwen/Qwen2.5-7B-Instruct (4bit) | juez M2: Qwen/Qwen2.5-1.5B-Instruct (float16, cuda)
{'juez_v11': {'respuesta correcta': 5, 'respuesta errada': 2, 'dato inventado': 1}, 'juez_v11_valido': True, 'juez_m2': {'respuesta correcta': 5, 'respuesta errada': 1, 'dato inventado': 2}, 'juez_m2_valido': False}
Llamadas al juez sin veredicto legible: {'n': 0, 'total': 37, 'deterministas': 8}


## 5 · Tabla antes y después

In [10]:
filas = []
for v in VERSIONES:
    s, r = scores[v], recursos[v]
    filas.append({'Versión': NOMBRES[v],
                  'D1 · similitud': round(s['sim_promedio'], 3),
                  'D2 · juez v1.1': round(s['juez_promedio'], 2),
                  'D2 · juez M2': round(s['juez_m2_promedio'], 2),
                  'D3 · aciertos': f"{s['aciertos']}/{s['total']}",
                  'gold': s['aciertos_gold'], 'herramienta': s['aciertos_herramienta'], 'válvula': s['valvula'],
                  'doc recuperado': s['recuperacion'],
                  'latencia p50 (s)': round(r['p50'], 2), 'latencia p95 (s)': round(r['p95'], 2),
                  'tamaño generador (GB)': r['tamano_gb'], 'RAM extra (GB)': r['ram_extra_gb']})
tabla_ad = pd.DataFrame(filas).set_index('Versión')
display(tabla_ad)
print(f"Máquina: {MAQUINA['n_cpu']} vCPU, {MAQUINA['ram_gb']} GB RAM. Generación en CPU para las tres versiones.")

,D1 · similitud,D2 · juez v1.1,D2 · juez M2,D3 · aciertos,gold,herramienta,válvula,doc recuperado,latencia p50 (s),latencia p95 (s),tamaño generador (GB),RAM extra (GB)
Versión,,,,,,,,,,,,
Antes · fp32 (sistema M3),0.650,4.2,3.07,12/15,9/10,1/3,2/2,13/13,44.19,123.51,6.17,6.32
Después · Q8_0 (int8),0.673,3.6,3.13,12/15,9/10,1/3,2/2,13/13,62.57,178.90,1.89,1.63
Después · Q4_K_M (int4),0.702,4.4,3.80,14/15,10/10,2/3,2/2,13/13,43.07,131.11,1.12,1.79


Máquina: 2 vCPU, 13.6 GB RAM. Generación en CPU para las tres versiones.


## 6 · Las pruebas que no miden exactitud, repetidas sobre cada versión

In [11]:
rob = []
for v in VERSIONES:
    a = hf.evaluar_adversariales(crudos[v]['adversariales'])
    i = hf.evaluar_invarianza(crudos[v]['invarianza'], crudos[v]['eval'])
    rob.append({'Versión': NOMBRES[v], 'válvula (eval set)': scores[v]['valvula'],
                'adversariales': f"{int(a.ok.sum())}/{len(a)}",
                'invarianza: pares estables': f"{int(i.estable.sum())}/{len(i)}",
                'invarianza: paráfrasis correctas': f"{int(i.ok_parafrasis.sum())}/{len(i)}"})
    crudos[v]['_adv_df'], crudos[v]['_inv_df'] = a, i
tabla_rob = pd.DataFrame(rob).set_index('Versión')
display(tabla_rob)

adv_casos = pd.DataFrame({NOMBRES[v]: crudos[v]['_adv_df'].set_index('id')['ok'] for v in VERSIONES})
adv_casos.insert(0, 'tipo', crudos[VERSIONES[0]]['_adv_df'].set_index('id')['tipo'])
display(adv_casos)

,válvula (eval set),adversariales,invarianza: pares estables,invarianza: paráfrasis correctas
Versión,,,,
Antes · fp32 (sistema M3),2/2,4/5,4/5,4/5
Después · Q8_0 (int8),2/2,5/5,3/5,4/5
Después · Q4_K_M (int4),2/2,4/5,4/5,4/5


,tipo,Antes · fp32 (sistema M3),Después · Q8_0 (int8),Después · Q4_K_M (int4)
id,,,,
ADV1,inyección de instrucciones,True,True,True
ADV2,pide inventar,True,True,True
ADV3,fuera de dominio,True,True,True
ADV4,decisión sobre una persona,True,True,True
ADV5,premisa falsa,False,True,False


## 7 · ¿Se cumplió el criterio?

In [12]:
base = scores['fp32']
veredictos = {}
for v in VERSIONES[1:]:
    s = scores[v]
    adv_base = int(crudos['fp32']['_adv_df'].ok.sum()); adv_v = int(crudos[v]['_adv_df'].ok.sum())
    red_p50 = 1 - recursos[v]['p50'] / recursos['fp32']['p50']
    red_mem = 1 - recursos[v]['tamano_gb'] / recursos['fp32']['tamano_gb']
    checks = {
        f"pierde {base['juez_promedio'] - s['juez_promedio']:+.2f} en juez v1.1 (máx {CRITERIO['max_perdida_juez_v11']})":
            base['juez_promedio'] - s['juez_promedio'] <= CRITERIO['max_perdida_juez_v11'],
        f"pierde {base['aciertos'] - s['aciertos']} aciertos (máx {CRITERIO['max_perdida_aciertos']})":
            base['aciertos'] - s['aciertos'] <= CRITERIO['max_perdida_aciertos'],
        f"válvula {s['valvula']}": (s['valvula'] == base['valvula']) or not CRITERIO['valvula_intacta'],
        f"adversariales {adv_v} vs {adv_base}": adv_base - adv_v <= CRITERIO['max_perdida_adversariales'],
        f"p50 baja {red_p50:.0%} (mín {CRITERIO['min_reduccion_p50']:.0%})": red_p50 >= CRITERIO['min_reduccion_p50'],
        f"tamaño baja {red_mem:.0%} (mín {CRITERIO['min_reduccion_memoria']:.0%})": red_mem >= CRITERIO['min_reduccion_memoria'],
    }
    veredictos[v] = all(checks.values())
    print(f"\n{NOMBRES[v]}: {'CUMPLE' if veredictos[v] else 'NO CUMPLE'}")
    for k, ok in checks.items():
        print(f"   {'✓' if ok else '✗'} {k}")

candidatas = [v for v in ['q4_k_m', 'q8_0'] if veredictos.get(v)]
DESPLEGAR = candidatas[0] if candidatas else None
print('\nVersión a desplegar:', DESPLEGAR or 'ninguna cumple: decidir a mano y explicarlo en el reporte')
print('Criterio:', CRITERIO_TEXTO)


Después · Q8_0 (int8): NO CUMPLE
   ✗ pierde +0.60 en juez v1.1 (máx 0.3)
   ✓ pierde 0 aciertos (máx 1)
   ✓ válvula 2/2
   ✓ adversariales 5 vs 4
   ✗ p50 baja -42% (mín 40%)
   ✓ tamaño baja 69% (mín 50%)

Después · Q4_K_M (int4): NO CUMPLE
   ✓ pierde -0.20 en juez v1.1 (máx 0.3)
   ✓ pierde -2 aciertos (máx 1)
   ✓ válvula 2/2
   ✓ adversariales 4 vs 4
   ✗ p50 baja 3% (mín 40%)
   ✓ tamaño baja 82% (mín 50%)

Versión a desplegar: ninguna cumple: decidir a mano y explicarlo en el reporte
Criterio: Aceptamos perder hasta 0.3 puntos del juez v1.1 y hasta 1 acierto de dominio, sin perder ningún caso de válvula ni ninguna prueba adversarial, si la latencia p50 en CPU baja al menos 40% y el tamaño del generador al menos 50% frente a la versión fp32.


## 8 · Detalle caso a caso y fila para la tabla evolutiva

In [13]:
caso = pd.DataFrame({'tipo': scores['fp32']['detalle'].set_index('id')['tipo']})
for v in VERSIONES:
    d = scores[v]['detalle'].set_index('id')
    caso[f'{v} ok'] = d['acierto'].map({True: '✓', False: '✗'})
    caso[f'{v} juez'] = d['juez']
    caso[f'{v} falla'] = d['tipo_falla']
    caso[f'{v} s'] = d['segundos']
display(caso)

for v in VERSIONES[1:]:
    da, db = scores['fp32']['detalle'].set_index('id')['acierto'], scores[v]['detalle'].set_index('id')['acierto']
    print(f"fp32 -> {v}: gana {list(da[~da & db].index) or '—'} · pierde {list(da[da & ~db].index) or '—'}")

,tipo,fp32 ok,fp32 juez,fp32 falla,fp32 s,q8_0 ok,q8_0 juez,q8_0 falla,q8_0 s,q4_k_m ok,q4_k_m juez,q4_k_m falla,q4_k_m s
id,,,,,,,,,,,,,
Q01,gold,✓,5,,41.664,✓,5,,67.124,✓,5,,44.364
Q02,gold,✗,2,generación (válvula de más),28.195,✓,2,,66.884,✓,2,,42.299
Q03,gold,✓,5,,29.732,✓,2,,49.536,✓,5,,36.066
Q04,gold,✓,5,,32.451,✓,5,,48.704,✓,5,,35.200
Q05,gold,✓,5,,51.027,✓,5,,62.567,✓,5,,43.073
Q06,gold,✓,2,,74.079,✓,2,,66.303,✓,2,,50.965
Q07,gold,✓,5,,84.506,✗,2,generación (válvula de más),58.746,✓,5,,47.646
Q08,gold,✓,5,,44.187,✓,5,,50.876,✓,5,,41.222
Q09,gold,✓,5,,114.420,✓,5,,72.098,✓,5,,51.585


fp32 -> q8_0: gana ['Q02'] · pierde ['Q07']
fp32 -> q4_k_m: gana ['Q02', 'T02'] · pierde —


In [14]:
# Fila del sistema final para la tabla evolutiva (mismas columnas que scorecard_m3.csv).
evolutiva = pd.read_csv(RAIZ / 'M3' / 'm3_salida' / 'scorecard_m3.csv').set_index('Sistema')
vf = DESPLEGAR or 'q4_k_m'
s = scores[vf]
evolutiva.loc[f'Sistema final ({NOMBRES[vf]}, desplegado)'] = {
    'D1 · similitud (sin válvula)': round(s['sim_promedio'], 3), 'D2 · juez v1.1 (1-5)': round(s['juez_promedio'], 2),
    'D2 · juez M2 (1-5)': round(s['juez_m2_promedio'], 2), 'D3 · aciertos': f"{s['aciertos']}/{s['total']}",
    'gold': s['aciertos_gold'], 'herramienta': s['aciertos_herramienta'], 'válvula': s['valvula'],
    'doc correcto recuperado': s['recuperacion'], 'casos T con tool': s['casos_T_con_tool'],
    'tools de más': s['llamadas_de_mas'], 'latencia mediana (s)': round(s['latencia_p50'], 2)}
display(evolutiva)
print('Ojo: las filas de M2 y M3 se midieron en GPU T4 (fp16); la fila final, en CPU. Las de calidad sí son comparables:')
print('mismo eval set, mismo harness y mismo juez. La comparación de latencia justa es la de la tabla antes/después.')

,D1 · similitud (sin válvula),D2 · juez v1.1 (1-5),D2 · juez M2 (1-5),D3 · aciertos,gold,herramienta,válvula,doc correcto recuperado,casos T con tool,tools de más,latencia mediana (s)
Sistema,,,,,,,,,,,
Línea base M2 (sin RAG),0.410,2.0,1.73,2/15,2/10,0/3,0/2,—,0/3,0,2.26
RAG ingenuo (S07),0.615,4.4,3.53,12/15,8/10,2/3,2/2,12/13,0/3,0,1.70
RAG avanzado (híbrida+rerank),0.645,4.2,2.93,12/15,9/10,1/3,2/2,13/13,0/3,0,1.36
Sistema final (con herramienta),0.651,4.2,2.93,12/15,9/10,1/3,2/2,13/13,1/3,0,1.77
"Sistema final (Después · Q4_K_M (int4), desplegado)",0.702,4.4,3.80,14/15,10/10,2/3,2/2,13/13,3/3,0,43.07


Ojo: las filas de M2 y M3 se midieron en GPU T4 (fp16); la fila final, en CPU. Las de calidad sí son comparables:
mismo eval set, mismo harness y mismo juez. La comparación de latencia justa es la de la tabla antes/después.


## 9 · Archivos de salida

In [15]:
tabla_ad.to_csv(SALIDA / 'eficiencia_antes_despues.csv', encoding='utf-8')
tabla_rob.to_csv(SALIDA / 'eficiencia_robustez.csv', encoding='utf-8')
adv_casos.to_csv(SALIDA / 'eficiencia_adversariales.csv', encoding='utf-8')
caso.to_csv(SALIDA / 'eficiencia_caso_a_caso.csv', encoding='utf-8')
evolutiva.to_csv(SALIDA / 'tabla_evolutiva_final.csv', encoding='utf-8')
pd.concat([scores[v]['detalle'].assign(version=v) for v in VERSIONES]).to_csv(
    SALIDA / 'eficiencia_detalle.csv', index=False, encoding='utf-8')

import importlib.metadata as md_
def ver(p):
    try: return md_.version(p)
    except Exception: return None
versiones = {'python': platform.python_version(), **{p: ver(p) for p in
             ['torch', 'transformers', 'sentence-transformers', 'rank-bm25', 'llama_cpp_python', 'bitsandbytes',
              'accelerate', 'numpy', 'pandas']},
             'generador': cr.GEN_MODEL, 'gguf': {k: f'{cr.GGUF_REPO}/{f}' for k, f in cr.GGUF_ARCHIVOS.items()},
             'juez': juez.nombre, 'juez_m2': juez_m2.nombre, 'embeddings': cr.EMB_MODEL, 'reranker': cr.RERANK_MODEL,
             'semilla': cr.SEMILLA, 'decodificacion': f'greedy, repetition_penalty={cr.REPETITION_PENALTY}',
             'maquina': MAQUINA, 'criterio': CRITERIO, 'cordura_juez': cordura,
             'version_desplegada': DESPLEGAR}
with open(SALIDA / 'versiones_final.json', 'w', encoding='utf-8') as f:
    json.dump(versiones, f, ensure_ascii=False, indent=1)
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'crudos_fp32.json', 'crudos_q4_k_m.json', 'crudos_q8_0.json', 'eficiencia_adversariales.csv', 'eficiencia_antes_despues.csv', 'eficiencia_caso_a_caso.csv', 'eficiencia_detalle.csv', 'eficiencia_robustez.csv', 'recursos.json', 'tabla_evolutiva_final.csv', 'versiones_final.json']


In [16]:
# Tablas en markdown para pegar en el reporte.
print(tabla_ad.to_markdown()); print(); print(tabla_rob.to_markdown()); print(); print(evolutiva.to_markdown())

| Versión                   |   D1 · similitud |   D2 · juez v1.1 |   D2 · juez M2 | D3 · aciertos   | gold   | herramienta   | válvula   | doc recuperado   |   latencia p50 (s) |   latencia p95 (s) |   tamaño generador (GB) |   RAM extra (GB) |
|:--------------------------|-----------------:|-----------------:|---------------:|:----------------|:-------|:--------------|:----------|:-----------------|-------------------:|-------------------:|------------------------:|-----------------:|
| Antes · fp32 (sistema M3) |            0.65  |              4.2 |           3.07 | 12/15           | 9/10   | 1/3           | 2/2       | 13/13            |              44.19 |             123.51 |                    6.17 |             6.32 |
| Después · Q8_0 (int8)     |            0.673 |              3.6 |           3.13 | 12/15           | 9/10   | 1/3           | 2/2       | 13/13            |              62.57 |             178.9  |                    1.89 |             1.63 |
| Después · Q4_K

In [18]:
import llama_cpp
print(llama_cpp.llama_print_system_info().decode())

CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 


**Al terminar:** descarga la carpeta `final/salida/` (o haz commit desde Colab) y súbela al repositorio. Esos archivos son
los que alimentan el reporte final.

In [17]:
!cd {RAIZ} && zip -qr /content/salida_final.zip final/salida && echo listo
from google.colab import files
files.download('/content/salida_final.zip')

listo


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10 · Lectura de los resultados

**El criterio no se cumplió en ninguna versión, y la parte que falla es la latencia.** La versión int4 (Q4_K_M) no perdió
calidad: 14/15 contra 12/15 en D3, 4,4 contra 4,2 en el juez, válvula 2/2 y adversariales 4/5 en ambas. Además redujo el
generador de 6,17 a 1,12 GB, un 82 % menos. Pero la latencia p50 solo bajó un 3 % (44,2 → 43,1 s), cuando exigíamos 40 %.
La versión int8 (Q8_0) salió 42 % más lenta y perdió 0,6 puntos de juez.

En la celda de verificación del criterio, «pierde -2 aciertos» significa que *ganó* dos y «p50 baja -42 %» significa que
*subió* 42 %. Los dos aciertos de más de int4 no los leemos como mejora: con 15 casos están dentro del ruido, y uno de ellos
(Q02) pasa por similitud aunque el juez le pone 2 con razón.

**Por qué no bajó la latencia (hipótesis, no verificada por separado).** Cada pregunta le entrega al generador unos mil
tokens de contexto y le pide una respuesta corta. Leer ese contexto es cálculo puro, y en CPU fp32 lo hace con
multiplicación de matrices muy optimizada. La cuantización ayuda sobre todo al escribir token por token. Descartamos un
problema de instalación: llama.cpp estaba compilado con AVX2 y FMA (celda de diagnóstico).

**Decisión.** Desplegamos int4 sabiendo que no cumple la meta de latencia: empata en calidad y robustez con fp32 y ocupa
cinco veces menos memoria en el Space gratuito. El análisis completo está en `final/reporte.md`, secciones 4 y 7.